In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import numpy as np
import time
import logging
import sys
import json
import copy
from datetime import datetime
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

# --- 设置日志和随机种子 ---
def setup_logging(log_file='grid_search.log'):
    log_formatter = logging.Formatter("%(asctime)s [%(levelname)-5.5s]  %(message)s")
    root_logger = logging.getLogger()
    root_logger.handlers = []
    root_logger.setLevel(logging.INFO)
    file_handler = logging.FileHandler(log_file, mode='w')
    file_handler.setFormatter(log_formatter)
    root_logger.addHandler(file_handler)
    console_handler = logging.StreamHandler()
    console_handler.setFormatter(log_formatter)
    root_logger.addHandler(console_handler)

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 数据准备函数 ---
def prepare_boston_data(test_size=0.2, random_state=42):
    """准备Boston房价数据集 - 80%训练, 20%验证"""
    logging.info("正在准备Boston房价数据集...")
    
    # 加载Boston房价数据集
    df = pd.read_csv('/kaggle/input/the-boston-houseprice-data/boston.csv')
    
    # 检查并处理数据
    if 'Unnamed: 0' in df.columns:
        df = df.drop('Unnamed: 0', axis=1)
    
    # 特征和目标变量
    X = df.drop(['MEDV'], axis=1).values  # 特征矩阵
    y = df['MEDV'].values  # 目标变量（房价中位数）
    
    # 数据标准化
    scaler_X = StandardScaler()
    X = scaler_X.fit_transform(X)
    
    # 对目标变量标准化
    scaler_y = StandardScaler()
    y = scaler_y.fit_transform(y.reshape(-1, 1)).flatten()
    
    # 划分训练验证集 (80%训练, 20%验证)
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    
    logging.info("Boston房价数据集准备完毕。")
    logging.info(f"训练样本数: {len(X_train)}, 验证样本数: {len(X_val)}")
    logging.info(f"特征维度: {X_train.shape[1]}, 目标维度: 1")
    
    return (X_train, y_train), (X_val, y_val), scaler_y

# --- 自定义数据集类 ---
class RegressionDataset(Dataset):
    """PyTorch数据集类"""
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float().view(-1, 1)
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# --- 模型架构 ---
class PolynomialTerm(nn.Module):
    """多项式项：输入 → 线性投影 → z → ∑ coeffs[i] * z^i"""
    def __init__(self, input_dim, output_dim, degree):
        super().__init__()
        self.degree = degree
        self.projection = nn.Linear(input_dim, output_dim)
        self.coeffs = nn.Parameter(torch.randn(output_dim, degree + 1) * 0.1)

    def forward(self, x):
        z = torch.tanh(self.projection(x))
        powers = [torch.ones_like(z)]
        for d in range(1, self.degree + 1):
            powers.append(powers[-1] * z)
        z_powered = torch.stack(powers, dim=-1)
        output = torch.sum(z_powered * self.coeffs, dim=2)
        return output

class CFNet(nn.Module):
    """单个 CFNet 子模型：continued-fraction 结构"""
    def __init__(self, input_dim, output_dim, depth, poly_degree):
        super().__init__()
        self.output_dim = output_dim
        self.terms = nn.ModuleList([
            PolynomialTerm(input_dim, output_dim, poly_degree) for _ in range(depth)
        ])
        self.raw_betas = nn.ParameterList([
            nn.Parameter(torch.full((output_dim,), 0.1)) for _ in range(depth - 1)
        ])

    def forward(self, x):
        if not self.terms:
            return torch.zeros(x.shape[0], self.output_dim, device=x.device)
        output = F.softplus(self.terms[-1](x)) + 1.0
        for i in range(len(self.terms) - 2, -1, -1):
            beta = F.softplus(self.raw_betas[i])
            output = self.terms[i](x) + beta / output
        return output

class EnsembleResCoFrNet(nn.Module):
    """集成模型：支持候选模型的选择性加入机制"""
    def __init__(self, input_dim, output_dim, shallow_depth, poly_degree, learning_rate):
        super().__init__()
        self.models = nn.ModuleList()          # 已采纳的正式模型
        self.candidate_model = None           # 正在评估的候选模型
        self.learning_rate = learning_rate
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.shallow_depth = shallow_depth
        self.poly_degree = poly_degree

    def forward(self, x):
        if not self.models and self.candidate_model is None:
            return torch.zeros(x.shape[0], self.output_dim, device=x.device)
            
        total_output = torch.zeros(x.shape[0], self.output_dim, device=x.device)
        
        # 累加已采纳的正式模型的输出
        for model in self.models:
            total_output += self.learning_rate * model(x)
            
        # 如果存在候选模型（在训练阶段），也累加其输出
        if self.candidate_model is not None:
            total_output += self.learning_rate * self.candidate_model(x)
            
        return total_output

    def create_candidate(self):
        """创建一个新的候选模型用于训练"""
        self.candidate_model = CFNet(
            input_dim=self.input_dim,
            output_dim=self.output_dim,
            depth=self.shallow_depth,
            poly_degree=self.poly_degree
        )
        return self.candidate_model

    def promote_candidate(self):
        """若候选模型有效，则将其采纳为正式模型"""
        if self.candidate_model is not None:
            self.models.append(self.candidate_model)
            self.candidate_model = None
    
    def discard_candidate(self):
        """丢弃无效的候选模型"""
        self.candidate_model = None

    def freeze_for_candidate_training(self):
        """冻结所有正式模型，只训练候选模型"""
        for model in self.models:
            for param in model.parameters():
                param.requires_grad = False
        if self.candidate_model is not None:
            for param in self.candidate_model.parameters():
                param.requires_grad = True

class CoFrNetRegressor:
    """
    一个集成了模型定义、训练、评估和保存/加载功能的高级封装器。
    """
    def __init__(self, hparams):
        """
        通过超参数字典初始化模型。
        :param hparams: 包含所有模型和训练所需参数的字典。
        """
        self.hparams = hparams
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        self.ensemble_model = EnsembleResCoFrNet(
            input_dim=self.hparams['input_dim'],
            output_dim=self.hparams['output_dim'],
            shallow_depth=self.hparams['shallow_depth_per_cofrnet'],
            poly_degree=self.hparams['polynomial_degree'],
            learning_rate=self.hparams['boosting_learning_rate']
        ).to(self.device)
        
        self.best_ensemble_state = None
        self.best_val_rmse = float('inf')
        self.best_model_size = 0
        logging.info(f"CoFrNetRegressor 初始化完毕，将在 {self.device} 上运行。")

    def train(self, train_data, val_data):
        """训练模型（使用选择性加入机制）"""
        X_train, y_train = train_data
        X_val, y_val = val_data

        train_dataset = RegressionDataset(X_train, y_train)
        val_dataset = RegressionDataset(X_val, y_val)

        trainloader = DataLoader(train_dataset, batch_size=self.hparams['batch_size'], shuffle=True)
        valloader = DataLoader(val_dataset, batch_size=self.hparams['batch_size'], shuffle=False)

        criterion = nn.MSELoss()
        patience = self.hparams.get('early_stopping_patience', 10)

        logging.info("="*30)
        logging.info(f"  开始 {self.hparams['task_name']} 回归任务训练 (选择性加入机制)")
        logging.info("="*30)
        
        # 外层循环：尝试添加新模型的次数
        for attempt in range(self.hparams['num_models_max']):
            logging.info(f"--- 尝试添加第 {len(self.ensemble_model.models) + 1} 个子模型 (总尝试次数: {attempt + 1}/{self.hparams['num_models_max']}) ---")
            
            # 1. 创建并训练一个候选模型
            candidate_model = self.ensemble_model.create_candidate().to(self.device)
            self.ensemble_model.freeze_for_candidate_training()

            optimizer = optim.Adam(
                candidate_model.parameters(), 
                lr=self.hparams['learning_rate_adam'], 
                weight_decay=self.hparams['weight_decay']
            )
            scheduler = optim.lr_scheduler.CosineAnnealingLR(
                optimizer, 
                T_max=self.hparams['epochs_per_model']
            )

            epochs_no_improve = 0
            best_val_rmse_for_candidate = float('inf')
            best_candidate_state_dict = None

            # 训练候选模型
            for epoch in range(self.hparams['epochs_per_model']):
                self.ensemble_model.train()
                running_loss = 0.0
                
                for inputs, targets in trainloader:
                    inputs, targets = inputs.to(self.device), targets.to(self.device)
                    optimizer.zero_grad()
                    outputs = self.ensemble_model(inputs)  # forward会自动包含候选模型
                    loss = criterion(outputs, targets)
                    loss.backward()
                    optimizer.step()
                    running_loss += loss.item()
                
                scheduler.step()
                
                # 评估包含候选模型的临时集成
                val_rmse = self.evaluate(valloader)
                
                if (epoch + 1) % 50 == 0:
                    epoch_avg_loss = running_loss / len(trainloader)
                    logging.info(f"    Epoch {epoch+1:03d} | 训练MSE损失: {epoch_avg_loss:.6f} | Val RMSE: {val_rmse:.6f}")

                # 早停检查
                if val_rmse < best_val_rmse_for_candidate:
                    best_val_rmse_for_candidate = val_rmse
                    epochs_no_improve = 0
                    best_candidate_state_dict = copy.deepcopy(candidate_model.state_dict())
                else:
                    epochs_no_improve += 1
                
                if epochs_no_improve >= patience:
                    logging.info(f"--- 候选模型早停触发！---")
                    if best_candidate_state_dict:
                        candidate_model.load_state_dict(best_candidate_state_dict)
                    break
            
            # 2. 评估这个训练好的候选模型加入后的效果
            logging.info("候选模型训练完毕，正在评估其对整体性能的贡献...")
            val_rmse_with_candidate = self.evaluate(valloader)
            logging.info(f"加入候选模型后，验证集 RMSE 为: {val_rmse_with_candidate:.6f}")
            logging.info(f"当前最佳模型的验证集 RMSE 为: {self.best_val_rmse:.6f}")

            # 3. 决策：是否采纳候选模型
            accepted = bool(val_rmse_with_candidate < self.best_val_rmse)
            
            if accepted:
                self.best_val_rmse = val_rmse_with_candidate
                self.ensemble_model.promote_candidate()
                self.best_model_size = len(self.ensemble_model.models)
                self.best_ensemble_state = copy.deepcopy(self.ensemble_model.state_dict())
                logging.info(f"*** 性能提升！采纳新模型。当前模型大小: {self.best_model_size}, 新的最佳 RMSE: {self.best_val_rmse:.6f} ***")
            else:
                self.ensemble_model.discard_candidate()
                logging.info(f"--- 性能未提升。丢弃该候选模型，继续尝试。---")

        # 训练结束，加载最佳模型状态
        if self.best_ensemble_state:
            logging.info(f"训练结束。最佳模型大小: {self.best_model_size}, RMSE: {self.best_val_rmse:.6f}")
            self.ensemble_model.load_state_dict(self.best_ensemble_state)
        else: 
            logging.error("没有训练出有效模型。")

        return self.best_val_rmse

    def evaluate(self, dataloader):
        """在给定数据集上评估模型并返回 RMSE"""
        self.ensemble_model.eval()
        total_mse = 0.0
        criterion = nn.MSELoss()
        
        with torch.no_grad():
            for inputs, targets in dataloader:
                inputs, targets = inputs.to(self.device), targets.to(self.device)
                outputs = self.ensemble_model(inputs)
                total_mse += criterion(outputs, targets).item() * inputs.size(0)
                
        rmse = np.sqrt(total_mse / len(dataloader.dataset))
        return rmse

# --- 网格搜索功能 ---
def grid_search_boston(depths, degrees, base_hparams, results_path='boston_grid_results.json'):
    """在Boston数据集上执行网格搜索"""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 准备数据
    train_data, val_data, scaler_y = prepare_boston_data()
    
    base_hparams['input_dim'] = train_data[0].shape[1]
    
    results = np.zeros((len(degrees), len(depths)), dtype=float)
    details = {}
    
    for i, deg in enumerate(degrees):
        for j, dep in enumerate(depths):
            logging.info(f"网格搜索点 >>> 阶数={deg}  深度={dep}")
            
            # 复制基础超参数并更新
            hp = copy.deepcopy(base_hparams)
            hp['polynomial_degree'] = deg
            hp['shallow_depth_per_cofrnet'] = dep
            
            # 创建回归器并训练
            reg = CoFrNetRegressor(hp)
            t0 = time.time()
            
            rmse = reg.train(train_data, val_data)
            
            elapsed = time.time() - t0
            
            # 记录结果
            results[i, j] = rmse
            details[f"{deg}_{dep}"] = {
                "rmse": float(rmse),
                "time": elapsed,
                "ensemble_size": reg.best_model_size
            }
            
            logging.info(f"<<< 阶数={deg} 深度={dep} | RMSE={rmse:.6f}  "
                         f"时间={elapsed:.1f}s  集成大小={reg.best_model_size}")
            
            # 清理内存
            del reg
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    
    # 保存结果
    with open(results_path, 'w') as f:
        json.dump({
            'depths': depths,
            'degrees': degrees,
            'results_matrix': results.tolist(),
            'details': details
        }, f, indent=2)
    
    return results, degrees, depths

def plot_boston_heatmap(results, degrees, depths, save_path='boston_heatmap_depth_vs_degree.png'):
    """绘制Boston数据集的热力图"""
    plt.figure(figsize=(max(6, len(depths)*0.6), max(4, len(degrees)*0.6)))
    sns.heatmap(results,
                xticklabels=depths, yticklabels=degrees,
                annot=True, fmt='.4f', cmap='viridis_r')  # 使用viridis_r以便低值显示为亮色
    plt.xlabel('Depth (shallow depth per CFNet)')
    plt.ylabel('Polynomial Degree')
    plt.title('Boston Housing CFNet-Boost Grid Search RMSE (lower is better)')
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    logging.info(f'Boston热力图已保存 -> {save_path}')

# --- 主程序 ---
if __name__ == '__main__':
    # 设置日志
    setup_logging('grid_search.log')
    
    # 基础超参数
    BASE_HPARAMS = {
        "task_name": "Boston Housing",
        "input_dim": 13,  
        "output_dim": 1, 
        "num_models_max": 15,  # 最大尝试添加模型次数
        "epochs_per_model": 100,  # 每个候选模型的最大训练轮数
        "shallow_depth_per_cofrnet": 2,  
        "polynomial_degree": 3,  
        "boosting_learning_rate": 0.1,  
        "learning_rate_adam": 1e-3,  
        "weight_decay": 1e-5,  
        "batch_size": 32,  
        "early_stopping_patience": 10,  
    }

    # 定义网格搜索范围
    depths = [1, 2, 3, 4, 5,6,7,8]
    degrees = [1, 2, 3, 4, 5,6,7,8]
    
    logging.info("开始Boston Housing网格搜索...")
    logging.info(f"深度范围: {depths}")
    logging.info(f"阶数范围: {degrees}")
    
    # 执行网格搜索
    results, degrees, depths = grid_search_boston(depths, degrees, BASE_HPARAMS)
    
    # 绘制热力图
    plot_boston_heatmap(results, degrees, depths)
    
    # 找到最佳参数
    best_idx = np.unravel_index(np.argmin(results), results.shape)
    best_degree = degrees[best_idx[0]]
    best_depth = depths[best_idx[1]]
    best_rmse = results[best_idx]
    
    logging.info(f"网格搜索完成！最佳参数: 阶数={best_degree}, 深度={best_depth}, RMSE={best_rmse:.6f}")
    
    print('Boston Housing网格搜索完成！')
    print(f'最佳参数: 阶数={best_degree}, 深度={best_depth}')
    print(f'最佳RMSE: {best_rmse:.6f}')